# Credit-Card Fraud Detection — Colab GPU Training

Trains the `TabularResNet` fraud model on the Kaggle **creditcard fraud** dataset
(284k transactions, 0.17% fraud) using the code from the `fraud-detection` branch
of [OmerBassan/IAI_TITANIC](https://github.com/OmerBassan/IAI_TITANIC).

**Before running:** set the runtime to GPU — *Runtime → Change runtime type → T4 GPU*.

Pipeline: clone repo → Kaggle data download → train on GPU → report PR-AUC + plots.

## 1. Confirm the GPU is attached

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU — go to Runtime > Change runtime type > T4 GPU, then rerun.')

## 2. Clone the project code (`fraud-detection` branch)

In [ ]:
%cd /content
![ -d IAI_TITANIC ] && rm -rf IAI_TITANIC
!git clone --branch fraud-detection --single-branch https://github.com/OmerBassan/IAI_TITANIC.git
%cd /content/IAI_TITANIC
!git log --oneline -1

## 3. Install dependencies
Colab already ships torch + sklearn + pandas; this just fills any gaps quietly.

In [ ]:
!pip install -q kaggle joblib optuna

## 4. Kaggle credentials
Upload your `kaggle.json` (Kaggle → Account → *Create New API Token*). It is
written to `~/.kaggle/` with the permissions the Kaggle CLI expects.

In [ ]:
import os, pathlib
from google.colab import files

if not pathlib.Path.home().joinpath('.kaggle/kaggle.json').exists():
    print('Upload kaggle.json:')
    files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp kaggle.json /root/.kaggle/kaggle.json
    !chmod 600 /root/.kaggle/kaggle.json
    print('kaggle.json installed.')
else:
    print('kaggle.json already present.')

## 5. Download + unzip the dataset
Pulls `mlg-ulb/creditcardfraud` into `data/creditcard.csv` (idempotent).

In [ ]:
import pathlib

csv_path = pathlib.Path('data/creditcard.csv')
if not csv_path.exists():
    !mkdir -p data
    !kaggle datasets download -d mlg-ulb/creditcardfraud -p data --unzip
else:
    print('creditcard.csv already present.')

import pandas as pd
df = pd.read_csv(csv_path)
print('shape:', df.shape)
print('fraud rate: {:.4f}%'.format(df['Class'].mean() * 100))

## 6. (Optional but recommended) Tune hyperparameters with Optuna
The first run came in *under* the RandomForest PR-AUC baseline (0.796), mainly
because the raw `pos_weight=599` inflated logits and hurt ranking. This search
maximises **validation PR-AUC** over lr, dropout, width, depth, weight-decay and
the `pos_weight` scale, then writes `fraud_best_params.json` — which the training
script in the next cell picks up automatically.

~30 trials on a T4 takes a few minutes. Skip this cell to train with the (already
improved) defaults instead.

In [ ]:
!python tuning_fraud.py --trials 30 --epochs 25 --device cuda

## 7. Train on GPU
Run via the CLI so it **auto-loads `fraud_best_params.json`** if the tuning cell
ran (otherwise it uses the improved defaults). Selection + early stopping are
keyed on **validation PR-AUC**; final numbers are reported on the held-out test
split with the frozen threshold.

In [ ]:
!python train_fraud.py --device cuda --epochs 60

## 8. Diagnostic plots (PR curve, ROC, confusion matrix, learning curves)
Re-runs the best checkpoint over the test split to draw the curves.

In [ ]:
import json, numpy as np, torch
from src.fraud_preprocessing import load_raw, make_splits, split_xy, load_preprocessor
from src.fraud_evaluation import (
    plot_pr_curve, plot_roc_curve, plot_confusion_matrix, plot_learning_curves,
)
from src.model import TabularResNet

meta = json.load(open('artifacts_fraud/metadata.json'))
thresh = meta['decision_threshold']

raw = load_raw('data/creditcard.csv')
_, _, test_df = make_splits(raw)
X_test_df, y_test = split_xy(test_df)
pre = load_preprocessor('artifacts_fraud/preprocessor.joblib')
X_test = pre.transform(X_test_df)

ckpt = torch.load('artifacts_fraud/model.pt', map_location='cpu', weights_only=False)
model = TabularResNet(**ckpt['arch'])
model.load_state_dict(ckpt['state_dict'])
with torch.no_grad():
    proba = model.predict_proba(torch.as_tensor(X_test, dtype=torch.float32)).numpy()
preds = (proba >= thresh).astype(int)

history = json.load(open('artifacts_fraud/history.json'))

plot_pr_curve(y_test.values, proba)
plot_roc_curve(y_test.values, proba)
plot_confusion_matrix(y_test.values, preds)
plot_learning_curves(history)
import matplotlib.pyplot as plt; plt.show()

## 9. (Optional) Save artifacts back to Google Drive
Persists the trained model + metadata so they survive the Colab session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p '/content/drive/MyDrive/fraud_artifacts'
!cp -r artifacts_fraud/* '/content/drive/MyDrive/fraud_artifacts/'
print('Saved to MyDrive/fraud_artifacts')